# 16 · Adaptive equalization

An adaptive equalizer learns the inverse channel from the data and keeps
following it as it changes, without a channel estimate
([15](15-linear-channel-equalization.ipynb)).  This notebook covers the data-aided and
decision-directed family:

1. **LMS**: a stochastic-gradient FIR filter, trained on known symbols and
   then driven by its own decisions;
2. the **step size**: convergence speed against steady-state error;
3. **RLS**: fast convergence, at a cost, and why it wants symbol-spaced
   input;
4. **block LMS**, the GPU-friendly variant;
5. carrier recovery **inside** the loop;
6. **continuing** on the next record with `state=`, and frozen taps.

The blind equalizers (CMA, RDE) follow in [17](17-blind-equalization.ipynb), and
the dual-polarization (2x2 MIMO) case in [18](18-polarization-multiplexing.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device
from commkit.recovery import BPS, CycleSlip

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)


def host(a):
    return to_device(a, "cpu")


def learning_curve(result, window=200):
    """Squared error per symbol, smoothed for display."""
    e2 = np.abs(host(result.error)) ** 2
    return ck.smoothing.moving_average(e2.reshape(-1, e2.shape[-1])[0], window=window)

## 1. The link and the LMS equalizer

16-QAM at 32 GBd and 2 samples per symbol through a 12 GHz electrical
bandwidth ([10](10-front-end-impairments.ipynb) · section 3), with receiver noise at Es/N0 = 20 dB
for the transmitted symbols, then the matched filter.  The noise is a fixed
floor (`noise_var`), so the power the filter removes counts against the SNR.  The ISI closes the eye completely.

`lms` runs a **fractionally spaced** FIR filter (here 21 taps at 2 samples
per symbol) and outputs one symbol per symbol period.  For each output `y`
it forms the error against a target `d` and nudges the taps down the
gradient of `|e|²`:

$$e = d - y, \qquad \mathbf{w} \leftarrow \mathbf{w} + \mu\, e\, \mathbf{x}^*.$$

During the first `len(training_symbols)` symbols `d` is the known symbol;
after that it is the nearest constellation point (decision-directed).  The
result is an `EqualizerResult`: the output `y_hat`, the final `weights`,
the per-symbol `error`, and for `Signal` input a 1-SPS `signal` with the
reference aligned to it.

In [ ]:
tx = ck.generate(
    C16,
    num_symbols=2**16,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=1,
    device=DEVICE,
)
# Receiver noise at Es/N0 = 20 dB for the transmitted unit-energy symbols
# (Es = 1 in sample units at any sps), fixed whatever the filter does.
NOISE_VAR = 10 ** (-20 / 10)
bessel = ck.filtering.bessel_sos(
    sampling_rate=tx.sampling_rate, cutoff=12e9, order=4, norm="mag"
)


def link(rng):
    x = ck.filtering.iir_filter(tx, bessel)
    x = ck.impairments.apply_awgn(x, noise_var=NOISE_VAR, rng=rng)
    return ck.filtering.matched_filter(x)


rx = link(2)
print(
    f"without equalizer: SNR {ck.metrics.snr(ck.multirate.decimate_to_symbol_rate(rx)):.1f} dB"
)

TRAIN = 4000
res = ck.equalization.lms(
    rx, num_taps=21, step_size=3e-3, training_symbols=rx.reference.symbols[..., :TRAIN]
)
print(
    f"LMS: SNR {ck.metrics.snr(res.signal, num_skip_symbols=2 * TRAIN):.2f} dB, "
    f"BER {ck.metrics.ber(res.signal, num_skip_symbols=2 * TRAIN):.1e}"
)
ck.plotting.plot_equalizer_result(res, show=True)
ck.plotting.plot_constellation(
    res.signal.samples[..., 2 * TRAIN :],
    constellation=C16,
    overlay_ideal=True,
    title="After LMS",
    show=True,
)

The tap plot shows what the equalizer learned: a main tap with
significant side taps that undo the Bessel response's spreading.  The
SNR stays below the 20 dB Es/N0 because the equalizer boosts the band edges
that the filter attenuated, and the noise there with them (the MMSE
trade-off of [15](15-linear-channel-equalization.ipynb), learned instead of computed).

## 2. Step size

The step size `μ` sets how far each update moves.  A large `μ` converges in
few symbols but keeps jittering around the optimum (the *misadjustment*,
an excess error proportional to `μ`), and beyond a limit set by the input
power and the number of taps it diverges (here between 0.04 and 0.08); a
small `μ` is precise but slow, and too small never converges within the
training.  After the training the error is measured against decisions,
which hides how wrong the slowest equalizer still is: its SNR tells.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for mu in (3e-4, 1e-3, 1e-2, 4e-2):
    r = ck.equalization.lms(
        rx,
        num_taps=21,
        step_size=mu,
        training_symbols=rx.reference.symbols[..., :TRAIN],
    )
    ax.semilogy(
        learning_curve(r),
        label=f"μ = {mu:g}: SNR {ck.metrics.snr(r.signal, num_skip_symbols=2 * TRAIN):.2f} dB",
    )
ax.axvline(TRAIN, color="gray", ls=":", label="end of training")
ax.set_xlabel("Symbol")
ax.set_ylabel("|e|² (moving average)")
ax.set_xlim(0, 20000)
ax.legend()
plt.show()

## 3. RLS

Recursive least squares minimizes the exponentially weighted squared error
over all past symbols exactly, by updating the inverse correlation matrix
of the input.  It converges in a few times the number of taps, almost
independently of the channel, at `O(num_taps²)` per symbol instead of
`O(num_taps)`.

RLS needs an invertible input correlation matrix.  At 2 samples per symbol
the band above the signal holds only noise, the matrix is nearly singular,
and the taps can grow without bound (`rls` warns about this).  So RLS runs
**symbol-spaced** here, at one sample per symbol, against LMS on the same
input.

In [ ]:
rx1 = ck.multirate.decimate_to_symbol_rate(rx)
fig, ax = plt.subplots(figsize=(8, 4))
for name, r in (
    (
        "LMS, μ = 3e-3",
        ck.equalization.lms(
            rx1,
            num_taps=11,
            step_size=3e-3,
            training_symbols=rx1.reference.symbols[..., :TRAIN],
        ),
    ),
    (
        "RLS, λ = 0.99",
        ck.equalization.rls(
            rx1,
            num_taps=11,
            forgetting_factor=0.99,
            training_symbols=rx1.reference.symbols[..., :TRAIN],
        ),
    ),
    (
        "RLS, λ = 0.999",
        ck.equalization.rls(
            rx1,
            num_taps=11,
            forgetting_factor=0.999,
            training_symbols=rx1.reference.symbols[..., :TRAIN],
        ),
    ),
):
    ax.semilogy(
        learning_curve(r, window=50),
        label=f"{name}: SNR {ck.metrics.snr(r.signal, num_skip_symbols=2 * TRAIN):.2f} dB",
    )
ax.set_xlim(0, 3000)
ax.set_xlabel("Symbol")
ax.set_ylabel("|e|² (moving average)")
ax.set_title("Symbol-spaced, 11 taps")
ax.legend()
plt.show()

The forgetting factor `λ` plays the role of `μ`: the effective memory is
`1/(1-λ)` symbols, so 0.99 tracks fast with more noise in the taps and
0.999 averages longer.

## 4. Block LMS

`lms` updates after every symbol, a sequential recursion that runs as
compiled code on the CPU.  `block_lms` holds the taps over a block of
`block_size` symbols, accumulates the gradient in the frequency domain and
updates once per block: the work inside a block is parallel, which is what
a GPU needs.  On the CPU, `lms` is usually faster (the function says so in
a log message).  The result has the same form.

In [ ]:
blk = ck.equalization.block_lms(
    rx, num_taps=21, block_size=256, training_symbols=rx.reference.symbols[..., :TRAIN]
)
print(f"block LMS: SNR {ck.metrics.snr(blk.signal, num_skip_symbols=2 * TRAIN):.2f} dB")

## 5. Carrier recovery inside the loop

With laser phase noise, the decisions of a decision-directed equalizer are
rotated, and its error mixes ISI with phase.  The CPR objects of
[13](13-blind-carrier-phase-recovery.ipynb) and [14](14-pilot-aided-carrier-recovery.ipynb) nest into the equalizer: `cpr=BPS(...)`
or `cpr=PLL(...)` estimates the phase on the equalizer's output inside the
loop, so taps and phase adapt together.  The training symbols train both,
so no rotation ambiguity remains.

In [ ]:
N = 2**15
stream = ck.generate(
    C16,
    num_symbols=2 * N,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=3,
    device=DEVICE,
)
stream = ck.impairments.apply_phase_noise(
    ck.filtering.iir_filter(stream, bessel), linewidth=1e6, rng=4
)
stream = ck.filtering.matched_filter(
    ck.impairments.apply_awgn(stream, noise_var=NOISE_VAR, rng=5)
)


def record(k):
    """Record k (0 or 1) of the stream, with its part of the reference."""
    ref = stream.reference
    return stream.replace(
        samples=stream.samples[..., 2 * k * N : 2 * (k + 1) * N],
        reference=ck.Reference(
            symbols=ref.symbols[..., k * N : (k + 1) * N],
            bits=ref.bits[..., 4 * k * N : 4 * (k + 1) * N],
        ),
    )


first = record(0)
cpr = BPS(test_phases=64, cycle_slip=CycleSlip(history=100))
plain = ck.equalization.lms(
    first,
    num_taps=21,
    step_size=3e-3,
    training_symbols=first.reference.symbols[..., :TRAIN],
)
inline = ck.equalization.lms(
    first,
    num_taps=21,
    step_size=3e-3,
    training_symbols=first.reference.symbols[..., :TRAIN],
    cpr=cpr,
)
print("1 MHz linewidth:")
for name, r in (("LMS alone", plain), ("LMS + inline BPS", inline)):
    print(
        f"  {name:17s}: SNR {ck.metrics.snr(r.signal, num_skip_symbols=2 * TRAIN):6.2f} dB"
    )
ck.plotting.plot_equalizer_result(inline, show=True)

Without CPR the decision-directed LMS tries to follow the phase with its
taps and cannot keep up with a 1 MHz linewidth; with BPS inside the loop the
taps only see the ISI.

## 6. The next record, and frozen taps

A capture is often processed in records.  `state=result.state` continues
exactly where the equalizer stopped: taps, carrier phase, normalization and
the samples still pending at the record's end.  The next record of the same
stream then needs no training and has no convergence transient.

`apply_taps(x, result.weights)` applies a set of taps without adapting
them: an inference pass, e.g. to measure what a converged equalizer does on
a different record.

In [ ]:
second = record(1)
cont = ck.equalization.lms(
    second, num_taps=21, step_size=3e-3, cpr=cpr, state=inline.state
)
print(
    f"record 2, continued, no training: SNR over the whole record {ck.metrics.snr(cont.signal):.2f} dB"
)

frozen = ck.equalization.apply_taps(rx, res.weights)
print(
    f"frozen taps of section 1 on its own record: SNR {ck.metrics.snr(frozen, num_skip_symbols=100):.2f} dB"
)

**Summary**

* LMS: train on known symbols, then follow decisions; `step_size` trades
  convergence speed against steady-state error.
* RLS converges in a few times `num_taps` symbols; run it symbol-spaced.
* `block_lms` is the GPU form; `lms` is faster on the CPU.
* `cpr=BPS()` / `cpr=PLL()` recovers the carrier inside the loop;
  `state=` continues on the next record.

**Next:** [17 · Blind equalization](17-blind-equalization.ipynb).